In [6]:
from pynq import Overlay, allocate
import numpy as np
import time

# 1. Load the Overlay
# Replace 'modulator.bit' with your actual bitfile name
ol = Overlay('dma_dac_bd.bit')

# 2. Initialize IP blocks
# Ensure these names match your Vivado Address Editor names
dma = ol.axi_dma_0
modulator = ol.ask_modulator_0

def run_transmission(binary_string):
    # Validation: Must be multiples of 8 bits
    if len(binary_string) % 8 != 0:
        print("Error: Input length must be a multiple of 8 bits.")
        return

    # Convert binary string to list of integers (bytes)
    # This takes every 8 characters and turns them into a uint8
    data_bytes = [int(binary_string[i:i+8], 2) for i in range(0, len(binary_string), 8)]
    
    # Allocate DMA buffer
    # We send one byte at a time to trigger the 3-second hardware timer per byte
    out_buffer = allocate(shape=(1,), dtype=np.uint8)

    print(f"--- Starting Transmission of {len(data_bytes)} byte(s) ---")
    
    for idx, byte_val in enumerate(data_bytes):
        # Format the byte back to string for printing
        byte_str = format(byte_val, '08b')
        
        # Explain the DSO display order (Right-to-Left processing in HLS)
        # Symbols: Bits [1:0], [3:2], [5:4], [7:6]
        s0 = byte_str[6:8]
        s1 = byte_str[4:6]
        s2 = byte_str[2:4]
        s3 = byte_str[0:2]
        
        print(f"\nSending Byte {idx + 1}: {byte_str}")
        print(f"DSO Display Order (Left to Right on screen):")
        print(f"  1st Symbol: {s0} | 2nd Symbol: {s1} | 3rd Symbol: {s2} | 4th Symbol: {s3}")
        print(f"Current Status: Displaying on DSO for 3 seconds...")

        # Prepare buffer and send
        out_buffer[0] = byte_val
        
        # Start the HLS IP (AXI-Lite Control)
        # 0x81 starts the IP and sets it to auto-restart if needed
        modulator.write(0x00, 0x01) 
        
        # Perform DMA transfer
        dma.sendchannel.transfer(out_buffer)
        dma.sendchannel.wait()
        
        # Wait 3 seconds in Python to match the hardware timer before sending next byte
       # time.sleep(3.1) 
        
    print("\n--- All data transmitted ---")

# --- USER INPUT ---
# You can input any binary string here (multiple of 8)
my_binary_data = "010101010101010101010101101010101010101010101010" 
run_transmission(my_binary_data)

--- Starting Transmission of 6 byte(s) ---

Sending Byte 1: 01010101
DSO Display Order (Left to Right on screen):
  1st Symbol: 01 | 2nd Symbol: 01 | 3rd Symbol: 01 | 4th Symbol: 01
Current Status: Displaying on DSO for 3 seconds...

Sending Byte 2: 01010101
DSO Display Order (Left to Right on screen):
  1st Symbol: 01 | 2nd Symbol: 01 | 3rd Symbol: 01 | 4th Symbol: 01
Current Status: Displaying on DSO for 3 seconds...

Sending Byte 3: 01010101
DSO Display Order (Left to Right on screen):
  1st Symbol: 01 | 2nd Symbol: 01 | 3rd Symbol: 01 | 4th Symbol: 01
Current Status: Displaying on DSO for 3 seconds...

Sending Byte 4: 10101010
DSO Display Order (Left to Right on screen):
  1st Symbol: 10 | 2nd Symbol: 10 | 3rd Symbol: 10 | 4th Symbol: 10
Current Status: Displaying on DSO for 3 seconds...

Sending Byte 5: 10101010
DSO Display Order (Left to Right on screen):
  1st Symbol: 10 | 2nd Symbol: 10 | 3rd Symbol: 10 | 4th Symbol: 10
Current Status: Displaying on DSO for 3 seconds...

Sendin

In [1]:
import time
import numpy as np
from pynq import Overlay, allocate

ol = Overlay('dma_dac_bd.bit')
dma = ol.axi_dma_0
modulator = ol.ask_modulator_0

# 1. Prepare a large dataset (e.g., 1 Million Bytes)
payload_size = 10**6 
test_data = allocate(shape=(payload_size,), dtype=np.uint8)
test_data[:] = np.random.randint(0, 255, payload_size, dtype=np.uint8)

def test_throughput():
    print(f"Transferring {payload_size / 1024 / 1024:.2f} MB...")
    
    # Start the Modulator IP (if not in auto-restart mode)
    modulator.write(0x00, 0x01) 
    
    start_time = time.time()
    
    # Perform the bulk transfer
    dma.sendchannel.transfer(test_data)
    dma.sendchannel.wait()
    
    end_time = time.time()
    
    duration = end_time - start_time
    throughput_mbps = (payload_size / (1024 * 1024)) / duration
    
    # Each byte results in 4 complex samples (128 bits)
    samples_per_sec = (payload_size * 4) / duration

    print(f"--- Throughput Results ---")
    print(f"Duration: {duration:.4f} seconds")
    print(f"PS-to-PL Speed: {throughput_mbps:.2f} MB/s")
    print(f"Effective Sample Rate: {samples_per_sec / 1e9:.4f} GSPS")

test_throughput()

Transferring 0.95 MB...


KeyboardInterrupt: 